# Comprehensive Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the emotion recognition dataset.

## Objectives:
1. Load and explore the balanced dataset
2. Comprehensive EDA with visualizations
3. Handle missing values
4. Detect and treat outliers
5. Address class imbalance using SMOTE
6. Engineer at least 3 new features
7. Justify preprocessing and feature engineering choices
8. Save the updated dataset

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from scipy import stats
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("All packages imported successfully!")

## 2. Load Dataset

In [ ]:
# Set paths
DATA_ROOT = Path('../data/processed/EmoSet_splits').resolve()
OUTPUT_ROOT = Path('../data/processed/EmoSet_splits_eda').resolve()
OUTPUT_ROOT.mkdir(exist_ok=True, parents=True)

# Load train, validation, and test sets
train_df = pd.read_csv(DATA_ROOT / 'train.csv')
val_df = pd.read_csv(DATA_ROOT / 'val.csv')
test_df = pd.read_csv(DATA_ROOT / 'test.csv')

# Combine for comprehensive analysis
df_all = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Total dataset size: {len(df_all)}")
print(f"\nColumns: {df_all.columns.tolist()}")
print(f"\nFirst few rows:")
df_all.head()

## 3. Basic Dataset Information

In [ ]:
# Dataset overview
print("Dataset Shape:", df_all.shape)
print("\nData Types:")
print(df_all.dtypes)
print("\nBasic Statistics:")
print(df_all.describe())
print("\nDataset Info:")
df_all.info()

## 4. Missing Value Analysis

In [ ]:
# Check for missing values
missing_values = df_all.isnull().sum()
missing_percent = (df_all.isnull().sum() / len(df_all)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage': missing_percent
})

print("Missing Values Analysis:")
print(missing_df[missing_df['Missing Count'] > 0])

if missing_df['Missing Count'].sum() == 0:
    print("\n✓ No missing values found in the dataset!")
else:
    print("\n⚠ Missing values detected and will be handled.")

# Visualize missing values
plt.figure(figsize=(10, 6))
sns.heatmap(df_all.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'missing_values_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Class Distribution Analysis (Before Balancing)

In [ ]:
# Class distribution
class_counts = df_all['label'].value_counts()
class_percentages = (df_all['label'].value_counts(normalize=True) * 100).round(2)

print("Class Distribution (Before Balancing):")
print("\nCounts:")
print(class_counts)
print("\nPercentages:")
print(class_percentages)

# Calculate imbalance ratio
max_class = class_counts.max()
min_class = class_counts.min()
imbalance_ratio = max_class / min_class
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}x")
print(f"Max class size: {max_class}")
print(f"Min class size: {min_class}")

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
class_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Emotion Class', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'class_distribution_before_balancing.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Extract Image Features for Analysis

Extract statistical features from images to perform outlier detection and feature engineering.

In [ ]:
def extract_image_features(img_path):
    """
    Extract statistical features from an image.
    
    Features extracted:
    - Mean intensity (overall and per channel)
    - Standard deviation (overall and per channel)
    - Min/Max pixel values
    - Aspect ratio
    - Image dimensions
    """
    try:
        # Convert path from container format to local format
        local_path = str(img_path).replace('/data/processed/', '../data/processed/')
        img = Image.open(local_path)
        img_array = np.array(img)
        
        # Basic statistics
        features = {
            'mean_intensity': np.mean(img_array),
            'std_intensity': np.std(img_array),
            'min_intensity': np.min(img_array),
            'max_intensity': np.max(img_array),
            'width': img.size[0],
            'height': img.size[1],
        }
        
        # Per-channel statistics (if RGB)
        if len(img_array.shape) == 3 and img_array.shape[2] == 3:
            features['mean_r'] = np.mean(img_array[:, :, 0])
            features['mean_g'] = np.mean(img_array[:, :, 1])
            features['mean_b'] = np.mean(img_array[:, :, 2])
            features['std_r'] = np.std(img_array[:, :, 0])
            features['std_g'] = np.std(img_array[:, :, 1])
            features['std_b'] = np.std(img_array[:, :, 2])
        
        return features
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return None

print("Extracting image features... This may take a few minutes.")
print(f"Processing {len(df_all)} images...")

# Sample a subset for faster processing (or use all)
USE_SAMPLE = True  # Set to False to process all images
SAMPLE_SIZE = 5000  # Sample size for faster processing

if USE_SAMPLE and len(df_all) > SAMPLE_SIZE:
    df_sample = df_all.sample(n=SAMPLE_SIZE, random_state=42).copy()
    print(f"Using a sample of {SAMPLE_SIZE} images for feature extraction.")
else:
    df_sample = df_all.copy()
    print(f"Processing all {len(df_sample)} images.")

# Extract features
features_list = []
for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    features = extract_image_features(row['path'])
    if features:
        features['index'] = idx
        features_list.append(features)

# Create features dataframe
features_df = pd.DataFrame(features_list)
features_df.set_index('index', inplace=True)

# Merge with original dataframe
df_sample = df_sample.join(features_df)

print(f"\nFeature extraction complete!")
print(f"Extracted features: {features_df.columns.tolist()}")
print(f"\nFeature statistics:")
df_sample[features_df.columns].describe()

## 7. Visualize Feature Distributions

In [ ]:
# Select numerical features for visualization
numerical_features = ['mean_intensity', 'std_intensity', 'mean_r', 'mean_g', 'mean_b', 
                      'std_r', 'std_g', 'std_b']

# Distribution plots
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, feature in enumerate(numerical_features):
    if feature in df_sample.columns:
        axes[idx].hist(df_sample[feature].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
        axes[idx].set_title(f'{feature} Distribution', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(feature, fontsize=10)
        axes[idx].set_ylabel('Frequency', fontsize=10)
        axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Outlier Detection and Treatment

We'll use two methods for outlier detection:
1. **IQR (Interquartile Range) Method**: Detects outliers beyond 1.5 * IQR
2. **Z-Score Method**: Detects outliers beyond 3 standard deviations

In [ ]:
def detect_outliers_iqr(df, column):
    """
    Detect outliers using IQR method.
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = (df[column] < lower_bound) | (df[column] > upper_bound)
    return outliers, lower_bound, upper_bound

def detect_outliers_zscore(df, column, threshold=3):
    """
    Detect outliers using Z-score method.
    """
    z_scores = np.abs(stats.zscore(df[column].dropna()))
    outliers = pd.Series([False] * len(df), index=df.index)
    outliers.loc[df[column].notna()] = z_scores > threshold
    return outliers

# Detect outliers for key features
outlier_features = ['mean_intensity', 'std_intensity']
outlier_summary = {}

for feature in outlier_features:
    if feature in df_sample.columns:
        iqr_outliers, lower, upper = detect_outliers_iqr(df_sample, feature)
        zscore_outliers = detect_outliers_zscore(df_sample, feature)
        
        outlier_summary[feature] = {
            'IQR_outliers': iqr_outliers.sum(),
            'IQR_lower': lower,
            'IQR_upper': upper,
            'Zscore_outliers': zscore_outliers.sum(),
            'Total_samples': len(df_sample)
        }

print("Outlier Detection Summary:")
for feature, summary in outlier_summary.items():
    print(f"\n{feature}:")
    print(f"  IQR Method: {summary['IQR_outliers']} outliers ({summary['IQR_outliers']/summary['Total_samples']*100:.2f}%)")
    print(f"  Z-Score Method: {summary['Zscore_outliers']} outliers ({summary['Zscore_outliers']/summary['Total_samples']*100:.2f}%)")
    print(f"  IQR Bounds: [{summary['IQR_lower']:.2f}, {summary['IQR_upper']:.2f}]")

In [ ]:
# Visualize outliers using box plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, feature in enumerate(outlier_features):
    if feature in df_sample.columns:
        bp = axes[idx].boxplot([df_sample[feature].dropna()], labels=[feature], patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
        axes[idx].set_title(f'{feature} - Outlier Detection (IQR)', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Value', fontsize=10)
        axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'outlier_detection_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

### Outlier Treatment Decision

**Justification**: 
- For emotion recognition, extreme brightness or contrast values may represent valid emotional expressions (e.g., very bright or very dark images may convey different emotions).
- Instead of removing outliers, we'll **cap them** at the 1st and 99th percentiles to preserve the dataset size while reducing extreme values.
- This approach maintains the natural variation in emotional expressions while preventing extreme outliers from skewing the model.

In [ ]:
# Treatment: Cap outliers at percentiles instead of removing them
def cap_outliers(df, column, lower_percentile=1, upper_percentile=99):
    """
    Cap outliers at specified percentiles.
    """
    lower_cap = df[column].quantile(lower_percentile / 100)
    upper_cap = df[column].quantile(upper_percentile / 100)
    df[column] = df[column].clip(lower=lower_cap, upper=upper_cap)
    return df, lower_cap, upper_cap

print("Applying outlier treatment (capping at 1st and 99th percentiles)...")

for feature in outlier_features:
    if feature in df_sample.columns:
        original_mean = df_sample[feature].mean()
        df_sample, lower_cap, upper_cap = cap_outliers(df_sample, feature)
        new_mean = df_sample[feature].mean()
        print(f"\n{feature}:")
        print(f"  Capped at: [{lower_cap:.2f}, {upper_cap:.2f}]")
        print(f"  Mean before: {original_mean:.2f}, after: {new_mean:.2f}")

print("\n✓ Outlier treatment complete!")

## 9. Feature Engineering

We'll create at least 3 new features that may help improve model performance:

1. **Brightness**: Overall brightness of the image (mean of all pixels)
2. **Contrast**: Measure of intensity variation (std of pixel values)
3. **Color Balance**: Ratio between channels to capture color dominance
4. **Edge Density**: Measure of edge content (approximated by standard deviation)
5. **RGB Variance**: Variance across RGB channels

In [ ]:
def engineer_features(df):
    """
    Engineer new features from existing image statistics.
    """
    # 1. Brightness: Overall brightness
    df['brightness'] = df['mean_intensity']
    
    # 2. Contrast: Measure of intensity variation
    df['contrast'] = df['std_intensity']
    
    # 3. Color Balance: Ratio of red to blue (captures warm vs cool tones)
    if 'mean_r' in df.columns and 'mean_b' in df.columns:
        df['color_balance_rb'] = df['mean_r'] / (df['mean_b'] + 1e-5)
    
    # 4. Edge Density: High std suggests more edges/details
    df['edge_density'] = df['std_intensity'] / (df['mean_intensity'] + 1e-5)
    
    # 5. RGB Variance: How much do the channels differ
    if all(col in df.columns for col in ['mean_r', 'mean_g', 'mean_b']):
        df['rgb_variance'] = df[['mean_r', 'mean_g', 'mean_b']].var(axis=1)
    
    # 6. Color Saturation: Difference between max and min channel
    if all(col in df.columns for col in ['mean_r', 'mean_g', 'mean_b']):
        df['color_saturation'] = df[['mean_r', 'mean_g', 'mean_b']].max(axis=1) - df[['mean_r', 'mean_g', 'mean_b']].min(axis=1)
    
    return df

# Apply feature engineering
df_sample = engineer_features(df_sample)

# List of new engineered features
engineered_features = ['brightness', 'contrast', 'color_balance_rb', 'edge_density', 'rgb_variance', 'color_saturation']

print("Engineered Features:")
for feature in engineered_features:
    if feature in df_sample.columns:
        print(f"  ✓ {feature}")

print(f"\nNew feature statistics:")
df_sample[engineered_features].describe()

In [ ]:
# Visualize engineered features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(engineered_features):
    if feature in df_sample.columns and idx < len(axes):
        axes[idx].hist(df_sample[feature].dropna(), bins=50, color='coral', alpha=0.7, edgecolor='black')
        axes[idx].set_title(f'{feature}', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(feature, fontsize=10)
        axes[idx].set_ylabel('Frequency', fontsize=10)
        axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'engineered_features.png', dpi=300, bbox_inches='tight')
plt.show()

### Feature Engineering Justification

**Why these features?**

1. **Brightness**: Emotional expressions can be associated with lighting conditions. Darker images might correlate with sad/angry emotions, while brighter images might correlate with happy emotions.

2. **Contrast**: High contrast images may indicate strong emotional expressions, while low contrast might indicate neutral expressions.

3. **Color Balance**: Color psychology suggests that warm colors (reds, oranges) are associated with energetic emotions (anger, happiness), while cool colors (blues) are associated with calm or sad emotions.

4. **Edge Density**: More edges/details might indicate more expressive faces with stronger emotions.

5. **RGB Variance**: Measures color diversity within the image, which can indicate environmental context that affects emotion perception.

6. **Color Saturation**: Highly saturated images might be perceived differently than desaturated ones in terms of emotional content.

## 10. Feature Correlation Analysis

In [ ]:
# Select features for correlation analysis
correlation_features = numerical_features + engineered_features
correlation_features = [f for f in correlation_features if f in df_sample.columns]

# Compute correlation matrix
correlation_matrix = df_sample[correlation_features].corr()

# Visualize correlation matrix
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'feature_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Find highly correlated features
high_corr_threshold = 0.8
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > high_corr_threshold:
            high_corr_pairs.append((
                correlation_matrix.columns[i], 
                correlation_matrix.columns[j], 
                correlation_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print(f"\nHighly correlated feature pairs (|r| > {high_corr_threshold}):")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
else:
    print(f"\nNo highly correlated features found (threshold: {high_corr_threshold})")

## 11. Feature Distribution by Class

In [ ]:
# Visualize feature distributions by emotion class
key_features = ['brightness', 'contrast', 'edge_density']

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for idx, feature in enumerate(key_features):
    if feature in df_sample.columns:
        df_sample.boxplot(column=feature, by='label', ax=axes[idx], patch_artist=True)
        axes[idx].set_title(f'{feature} by Emotion Class', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Emotion', fontsize=10)
        axes[idx].set_ylabel(feature, fontsize=10)
        axes[idx].tick_params(axis='x', rotation=45)
        plt.setp(axes[idx].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'features_by_emotion_class.png', dpi=300, bbox_inches='tight')
plt.show()

## 12. Prepare Features for SMOTE

Before applying SMOTE, we need to prepare the feature matrix and handle the full dataset.

In [ ]:
# For SMOTE, we need to apply feature extraction to the entire dataset
# Since this is computationally expensive, we'll save the feature dataframe

# Define a function to process the full dataset in batches
def process_full_dataset_features(df, batch_size=1000):
    """
    Process full dataset in batches to extract features.
    """
    all_features = []
    
    for i in tqdm(range(0, len(df), batch_size), desc="Processing batches"):
        batch = df.iloc[i:i+batch_size]
        batch_features = []
        
        for idx, row in batch.iterrows():
            features = extract_image_features(row['path'])
            if features:
                features['index'] = idx
                batch_features.append(features)
        
        all_features.extend(batch_features)
    
    features_df = pd.DataFrame(all_features)
    features_df.set_index('index', inplace=True)
    
    return features_df

# Check if we need to process all data or use sample
if USE_SAMPLE:
    print(f"Using sampled dataset with {len(df_sample)} images for SMOTE.")
    df_for_smote = df_sample.copy()
else:
    print(f"Processing full dataset with {len(df_all)} images...")
    print("This may take 10-20 minutes. Please wait...")
    full_features_df = process_full_dataset_features(df_all)
    df_for_smote = df_all.join(full_features_df)
    df_for_smote = engineer_features(df_for_smote)
    print("✓ Full dataset feature extraction complete!")

# Prepare feature matrix
feature_columns = [col for col in df_for_smote.columns 
                   if col not in ['path', 'class_name', 'label_id', 'split', 'label', 'width', 'height']]
feature_columns = [col for col in feature_columns if df_for_smote[col].notna().all()]

print(f"\nFeatures to use for SMOTE: {feature_columns}")
print(f"Number of features: {len(feature_columns)}")

## 13. Apply SMOTE for Class Balancing

**Justification for SMOTE**:
- SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples for minority classes by interpolating between existing samples.
- This helps prevent overfitting compared to simple duplication.
- It maintains the feature space characteristics while balancing the classes.

In [ ]:
# Prepare data for SMOTE
# Drop rows with missing values in feature columns
df_clean = df_for_smote.dropna(subset=feature_columns).copy()

print(f"Dataset shape before SMOTE: {df_clean.shape}")
print(f"Class distribution before SMOTE:")
print(df_clean['label'].value_counts())

# Prepare X and y
X = df_clean[feature_columns].values
y = df_clean['label'].values

# Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"\nLabel mapping:")
for i, label in enumerate(le.classes_):
    print(f"  {label}: {i}")

# Apply SMOTE
print("\nApplying SMOTE...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_balanced, y_balanced = smote.fit_resample(X, y_encoded)

# Decode labels back
y_balanced_labels = le.inverse_transform(y_balanced)

print(f"\nDataset shape after SMOTE: {X_balanced.shape}")
print(f"Class distribution after SMOTE:")
class_dist_after = pd.Series(y_balanced_labels).value_counts()
print(class_dist_after)

# Create balanced dataframe
df_balanced = pd.DataFrame(X_balanced, columns=feature_columns)
df_balanced['label'] = y_balanced_labels

print("\n✓ SMOTE balancing complete!")

## 14. Visualize Class Distribution After Balancing

In [ ]:
# Compare class distributions before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before SMOTE
class_dist_before = df_clean['label'].value_counts()
class_dist_before.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Class Distribution BEFORE SMOTE', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Emotion Class', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# After SMOTE
class_dist_after.plot(kind='bar', ax=axes[1], color='forestgreen')
axes[1].set_title('Class Distribution AFTER SMOTE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Emotion Class', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'class_distribution_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate balancing metrics
balance_ratio_before = class_dist_before.max() / class_dist_before.min()
balance_ratio_after = class_dist_after.max() / class_dist_after.min()

print(f"\nBalancing Metrics:")
print(f"  Before SMOTE - Max/Min ratio: {balance_ratio_before:.2f}x")
print(f"  After SMOTE - Max/Min ratio: {balance_ratio_after:.2f}x")
print(f"  Improvement: {(balance_ratio_before - balance_ratio_after):.2f}x reduction in imbalance")
print(f"\n  Total samples before: {len(df_clean)}")
print(f"  Total samples after: {len(df_balanced)}")
print(f"  Synthetic samples generated: {len(df_balanced) - len(df_clean)}")

## 15. Dataset Overview After Balancing

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Class distribution comparison
ax1 = fig.add_subplot(gs[0, :])
x = np.arange(len(class_dist_before))
width = 0.35
ax1.bar(x - width/2, class_dist_before.values, width, label='Before SMOTE', color='steelblue')
ax1.bar(x + width/2, class_dist_after.values, width, label='After SMOTE', color='forestgreen')
ax1.set_xlabel('Emotion Class', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Class Distribution Comparison: Before vs After SMOTE', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(class_dist_before.index, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2-4. Feature distributions after balancing
key_features_plot = ['brightness', 'contrast', 'edge_density']
for idx, feature in enumerate(key_features_plot):
    if feature in df_balanced.columns:
        ax = fig.add_subplot(gs[1, idx])
        ax.hist(df_balanced[feature], bins=50, color='coral', alpha=0.7, edgecolor='black')
        ax.set_title(f'{feature} Distribution (Balanced)', fontsize=11, fontweight='bold')
        ax.set_xlabel(feature, fontsize=10)
        ax.set_ylabel('Frequency', fontsize=10)
        ax.grid(axis='y', alpha=0.3)

# 5-7. Feature by class after balancing
for idx, feature in enumerate(key_features_plot):
    if feature in df_balanced.columns:
        ax = fig.add_subplot(gs[2, idx])
        df_balanced.boxplot(column=feature, by='label', ax=ax)
        ax.set_title(f'{feature} by Class (Balanced)', fontsize=11, fontweight='bold')
        ax.set_xlabel('Emotion', fontsize=10)
        ax.set_ylabel(feature, fontsize=10)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Dataset Overview After Balancing with SMOTE', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(OUTPUT_ROOT / 'dataset_overview_after_balancing.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Dataset overview visualization complete!")

## 16. Save Processed Dataset

Save the balanced dataset with engineered features for downstream model training.

In [ ]:
# Save balanced dataset
output_file = OUTPUT_ROOT / 'balanced_dataset_with_features.csv'
df_balanced.to_csv(output_file, index=False)
print(f"✓ Balanced dataset saved to: {output_file}")
print(f"  Shape: {df_balanced.shape}")
print(f"  Features: {df_balanced.columns.tolist()}")

# Save feature list for reference
feature_info = {
    'numerical_features': numerical_features,
    'engineered_features': engineered_features,
    'all_features': feature_columns,
    'label_encoding': {label: int(idx) for idx, label in enumerate(le.classes_)}
}

feature_info_file = OUTPUT_ROOT / 'feature_info.json'
with open(feature_info_file, 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f"✓ Feature information saved to: {feature_info_file}")

# Save summary statistics
summary = {
    'original_dataset_size': len(df_clean),
    'balanced_dataset_size': len(df_balanced),
    'synthetic_samples_generated': len(df_balanced) - len(df_clean),
    'num_features': len(feature_columns),
    'num_classes': len(class_dist_after),
    'class_distribution_before': class_dist_before.to_dict(),
    'class_distribution_after': class_dist_after.to_dict(),
    'balance_ratio_before': float(balance_ratio_before),
    'balance_ratio_after': float(balance_ratio_after)
}

summary_file = OUTPUT_ROOT / 'eda_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ EDA summary saved to: {summary_file}")

## 17. Preprocessing and Feature Engineering Justification Summary

### Data Quality & Preparation
1. **Missing Values**: No missing values were found in the dataset paths and labels. Image features were extracted successfully.

2. **Outlier Treatment**: Instead of removing outliers, we capped them at the 1st and 99th percentiles. This preserves dataset size while preventing extreme values from negatively impacting the model. This is particularly important for emotion recognition as extreme lighting/contrast may represent valid emotional states.

### Feature Engineering Choices
1. **Brightness**: Captures overall illumination, which can correlate with emotional content
2. **Contrast**: Measures intensity variation, potentially indicating emotional expression strength
3. **Color Balance**: Captures warm vs cool color dominance, linked to emotional psychology
4. **Edge Density**: Approximates facial detail/expression intensity
5. **RGB Variance**: Measures color diversity, capturing environmental context
6. **Color Saturation**: Differentiates vivid vs muted color palettes

### Class Imbalance Handling
- **SMOTE** was applied to balance classes by generating synthetic samples through interpolation
- This approach:
  - Prevents overfitting to minority classes
  - Maintains feature space characteristics
  - Creates more diverse training examples than simple duplication
  - Achieved near-perfect balance across all emotion classes

### Next Steps for Model Development
The balanced dataset with engineered features is ready for:
1. Model training (CNN, Transfer Learning)
2. Feature selection/importance analysis
3. Hyperparameter tuning with balanced classes
4. Performance evaluation with better representation of all emotions

## 18. Generate Final Report

In [ ]:
# Generate markdown report
report = f"""
# Exploratory Data Analysis Report

## Dataset Overview
- **Original Dataset Size**: {len(df_clean):,} samples
- **Balanced Dataset Size**: {len(df_balanced):,} samples
- **Synthetic Samples Generated**: {len(df_balanced) - len(df_clean):,}
- **Number of Features**: {len(feature_columns)}
- **Number of Classes**: {len(class_dist_after)}

## Class Distribution

### Before SMOTE
{class_dist_before.to_string()}

**Imbalance Ratio**: {balance_ratio_before:.2f}x

### After SMOTE
{class_dist_after.to_string()}

**Imbalance Ratio**: {balance_ratio_after:.2f}x

## Features

### Extracted Features
{', '.join(numerical_features)}

### Engineered Features
{', '.join(engineered_features)}

## Key Findings
1. Dataset was successfully balanced using SMOTE
2. {len(engineered_features)} new features were engineered
3. Outliers were treated by capping at percentiles
4. No missing values in the processed dataset
5. All visualizations saved to `{OUTPUT_ROOT}`

## Output Files
- `balanced_dataset_with_features.csv`: Balanced dataset with all features
- `feature_info.json`: Feature metadata and label encoding
- `eda_summary.json`: Complete EDA summary statistics
- Multiple visualization PNG files

---
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

report_file = OUTPUT_ROOT / 'EDA_REPORT.md'
with open(report_file, 'w') as f:
    f.write(report)

print(f"✓ Final report saved to: {report_file}")
print("\n" + "="*80)
print("EDA COMPLETE!")
print("="*80)
print(f"\nAll outputs saved to: {OUTPUT_ROOT}")
print(f"\nKey files:")
print(f"  - {output_file.name}")
print(f"  - {feature_info_file.name}")
print(f"  - {summary_file.name}")
print(f"  - {report_file.name}")
print(f"\nVisualization files: {len(list(OUTPUT_ROOT.glob('*.png')))} PNG images")